# BLIP-2 query-count experiment

Run this notebook with **Python (blip2 AMD)** from the project root or `scripts/`.
It reuses `load_blip2.ipynb`, the existing BF16 precision, automatic device placement,
12 GiB placement budget, AMD ROCm environment, and shared model cache.

The smoke stage evaluates 10 questions in each of 13 conditions. The pilot and main
stages use the first 500 and all 2,000 questions of the existing seed-42 subset.
Five fixed, nested query subsets are used for each pruning condition. K=64 and
K=128 repeat pretrained query embeddings before the Q-Former and are exploratory
controls. All conditions use five-beam decoding. The single-question greedy
baseline notebook serves as a separate functionality check.

Results are saved after every question. Rerun with identical settings to resume.
Run `score_results.ipynb` after this notebook completes. Stop other kernels holding
this model before running an independent kernel.


In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from pathlib import Path
import hashlib
import json
import platform
import random
import time
import uuid
from datetime import datetime, timezone
from importlib.metadata import version

import nbformat
import torch
from filelock import FileLock
from PIL import Image
from tqdm.auto import tqdm

STAGE = os.environ.get("BLIP2_STAGE", "smoke")  # smoke, pilot, or main
QUESTION_COUNTS = {"smoke": 10, "pilot": 500, "main": 2000}
QUERY_SEEDS = [0, 1, 2, 3, 4]
RUN_ORDER_SEED = 2026
WARMUP_QUESTIONS = 2
PIPELINE_VERSION = 1
PROMPT = "Question: {question} Short answer:"
GENERATION_SETTINGS = {
    "do_sample": False,
    "num_beams": 5,
    "length_penalty": -1.0,
    "max_new_tokens": 10,
}

if STAGE not in QUESTION_COUNTS:
    raise ValueError("STAGE must be smoke, pilot, or main.")
if len(QUERY_SEEDS) != len(set(QUERY_SEEDS)) or not QUERY_SEEDS:
    raise ValueError("Query seeds must be unique and nonempty.")

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "scripts" else cwd
DATA_DIR = PROJECT_ROOT / "data" / "validation_dataset"
IMAGE_DIR = DATA_DIR / "images" / "val2014"
RESULTS_ROOT = PROJECT_ROOT / "results" / "query_ablation"


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".partial")
    try:
        with temporary.open("w", encoding="utf-8") as stream:
            json.dump(value, stream, indent=2, allow_nan=False)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def image_path(question):
    return IMAGE_DIR / f"COCO_val2014_{int(question['image_id']):012d}.jpg"


In [ ]:
questions_path = DATA_DIR / "v2_OpenEnded_mscoco_val2014_questions_subset.json"
annotations_path = DATA_DIR / "v2_mscoco_val2014_annotations_subset.json"
all_questions = json.loads(questions_path.read_text(encoding="utf-8"))["questions"]
annotations = json.loads(annotations_path.read_text(encoding="utf-8"))["annotations"]
annotation_by_id = {int(item["question_id"]): item for item in annotations}
if len(annotation_by_id) != len(annotations):
    raise ValueError("Duplicate annotation question IDs.")
if len(all_questions) != 2000:
    raise ValueError("Expected the existing 2,000-question validation subset.")
questions = all_questions[:QUESTION_COUNTS[STAGE]]
question_ids = [int(question["question_id"]) for question in questions]
if len(set(question_ids)) != len(question_ids):
    raise ValueError("Duplicate question IDs in the selected data.")
for question in questions:
    annotation = annotation_by_id[int(question["question_id"])]
    if int(annotation["image_id"]) != int(question["image_id"]):
        raise ValueError("Question and annotation image IDs differ.")
    if len(annotation["answers"]) != 10:
        raise ValueError("Official VQA scoring requires all ten annotator answers.")
    if not image_path(question).is_file():
        raise FileNotFoundError(image_path(question))

image_hashes = {path.name: file_sha256(path) for path in sorted({image_path(q) for q in questions})}
print(f"Stage {STAGE}: {len(questions)} questions, {len(image_hashes)} distinct images.")


Stage smoke: 10 questions, 10 distinct images.


In [ ]:
# Restore the canonical queries before rerunning the shared loader in this kernel.
if "ablation_original_parameter" in globals():
    model.query_tokens = ablation_original_parameter
    model.config.num_query_tokens = 32
    processor.num_query_tokens = 32

loader = PROJECT_ROOT / "scripts" / "load_blip2.ipynb"
get_ipython().run_line_magic("run", f'-i "{loader.as_posix()}"')
if model.query_tokens.shape != (1, 32, 768):
    raise RuntimeError("Start from the original 32-query checkpoint in a fresh kernel.")
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("This experiment requires the verified BF16 GPU environment.")

ablation_original_parameter = model.query_tokens
ablation_original_parameter.requires_grad_(False)
original_queries = ablation_original_parameter.detach().clone()
model.requires_grad_(False)
model.eval()
vision_parameter = next(model.vision_model.parameters())
DEVICE = vision_parameter.device
DTYPE = vision_parameter.dtype
if DEVICE.type != "cuda":
    raise RuntimeError("The vision encoder must be on the GPU for these measurements.")


def query_indices(k, seed):
    if k not in (8, 16, 32, 64, 128):
        raise ValueError("Unsupported query count.")
    if k < 32:
        generator = torch.Generator().manual_seed(seed)
        return torch.randperm(32, generator=generator)[:k].sort().values
    return torch.arange(32).repeat(k // 32)


def set_query_count(k, seed=None):
    indices = query_indices(k, seed)
    selected = original_queries.index_select(1, indices.to(original_queries.device)).clone()
    model.query_tokens = torch.nn.Parameter(selected, requires_grad=False)
    model.config.num_query_tokens = k
    processor.num_query_tokens = k
    assert model.query_tokens.shape == (1, k, 768)
    return indices.tolist()


def restore_original_queries():
    model.query_tokens = ablation_original_parameter
    model.config.num_query_tokens = 32
    processor.num_query_tokens = 32


def condition_name(k, seed):
    return f"k{k:03d}-seed{seed:03d}" if seed is not None else f"k{k:03d}"


conditions = [(k, seed) for k in (8, 16) for seed in QUERY_SEEDS]
conditions += [(32, None), (64, None), (128, None)]
random.Random(RUN_ORDER_SEED).shuffle(conditions)
print("GPU:", torch.cuda.get_device_name(DEVICE))
print("Precision:", DTYPE, "| device placement:", model.hf_device_map)
print("Run order:", [condition_name(k, seed) for k, seed in conditions])


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

BLIP-2 ready (32 query tokens).
Model architecture:
['Blip2ForConditionalGeneration']

Number of pretrained queries:
32

Query embedding shape:
(1, 32, 768)

Q-Former hidden dimension:
768

Vision encoder hidden dimension:
1408

Vision encoder layers:
39
GPU: AMD Radeon RX 9070
Precision: torch.bfloat16 | device placement: {'': 0}
Run order: ['k008-seed000', 'k016-seed004', 'k008-seed002', 'k016-seed002', 'k064', 'k016-seed001', 'k008-seed004', 'k008-seed003', 'k128', 'k032', 'k016-seed003', 'k016-seed000', 'k008-seed001']


## Measurement protocol

Batch size is one. Each condition receives two unmeasured warmup questions.
Generation time covers `model.generate()` with GPU synchronization on both sides.
Image reading, preprocessing, transfers, decoding, and result writing are excluded.
Throughput is the number of measured questions divided by summed generation time.
Peak allocated and reserved GPU memory are reset for every question, after inputs
have been transferred. Allocated memory includes the resident model and inputs.
Generated length excludes the decoder start token and padding, and includes EOS
when emitted. Fixed seeds govern query selection and condition order.

These measurements apply to the hardware and device map recorded in the manifest.
Compare your group mate's speed and memory results within their hardware setup.


In [ ]:
checkpoint_revision = getattr(model.config, "_commit_hash", None)
if not checkpoint_revision:
    raise RuntimeError("Could not identify the loaded Hugging Face checkpoint revision.")
notebook = nbformat.read(PROJECT_ROOT / "scripts" / "query_ablation.ipynb", as_version=4)
code_digest = hashlib.sha256("\n".join(cell.source for cell in notebook.cells if cell.cell_type == "code").encode()).hexdigest()
packages = ["torch", "torchvision", "transformers", "accelerate", "huggingface_hub", "numpy", "Pillow"]
experiment = {
    "pipeline_version": PIPELINE_VERSION,
    "notebook_code_sha256": code_digest,
    "loader_sha256": file_sha256(loader),
    "stage": STAGE,
    "model_id": MODEL_ID,
    "checkpoint_revision": checkpoint_revision,
    "original_query_sha256": hashlib.sha256(original_queries.cpu().contiguous().view(torch.uint8).numpy().tobytes()).hexdigest(),
    "question_file_sha256": file_sha256(questions_path),
    "annotation_file_sha256": file_sha256(annotations_path),
    "question_ids": question_ids,
    "image_sha256": image_hashes,
    "prompt": PROMPT,
    "generation": GENERATION_SETTINGS,
    "batch_size": 1,
    "warmup_questions": WARMUP_QUESTIONS,
    "precision": str(DTYPE),
    "device_map": {key: str(value) for key, value in model.hf_device_map.items()},
    "gpu_name": torch.cuda.get_device_name(DEVICE),
    "gpu_memory_bytes": torch.cuda.get_device_properties(DEVICE).total_memory,
    "rocm_version": torch.version.hip,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "packages": {name: version(name) for name in packages},
    "run_order_seed": RUN_ORDER_SEED,
    "query_seeds": QUERY_SEEDS,
    "conditions": [
        {"name": condition_name(k, seed), "k": k, "seed": seed,
         "query_indices": query_indices(k, seed).tolist(),
         "kind": "pruning" if k < 32 else "baseline" if k == 32 else "repeated-query control"}
        for k, seed in conditions
    ],
}
signature = hashlib.sha256(json.dumps(experiment, sort_keys=True).encode()).hexdigest()
EXPERIMENT_DIR = RESULTS_ROOT / f"{STAGE}-{signature[:16]}"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
print("Results directory:", EXPERIMENT_DIR)


In [ ]:
def read_records(path, expected_ids, condition):
    """Recover an interrupted final write and validate the saved ordered prefix."""
    if not path.exists():
        return []
    records = []
    with path.open("rb+") as stream:
        while True:
            start = stream.tell()
            line = stream.readline()
            if not line:
                break
            if not line.endswith(b"\n"):
                stream.truncate(start)
                stream.flush()
                os.fsync(stream.fileno())
                break
            record = json.loads(line)
            position = len(records)
            if position >= len(expected_ids) or record["question_id"] != expected_ids[position]:
                raise ValueError(f"Unexpected or duplicate question in {path}.")
            if record["k"] != condition["k"] or record["query_seed"] != condition["seed"]:
                raise ValueError(f"Saved condition differs in {path}.")
            if not isinstance(record["answer"], str) or record["generation_seconds"] <= 0:
                raise ValueError(f"Invalid prediction record in {path}.")
            records.append(record)
    return records


def generate_answer(question):
    with Image.open(image_path(question)) as image:
        inputs = processor(
            images=image.convert("RGB"),
            text=PROMPT.format(question=question["question"]),
            return_tensors="pt",
        )
    placeholder_count = int((inputs["input_ids"] == model.config.image_token_index).sum())
    if placeholder_count != model.query_tokens.shape[1]:
        raise RuntimeError("Image placeholder count differs from the current query count.")
    inputs = inputs.to(device=DEVICE, dtype=DTYPE)
    torch.cuda.synchronize(DEVICE)
    torch.cuda.reset_peak_memory_stats(DEVICE)
    allocated_before = torch.cuda.memory_allocated(DEVICE)
    started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**inputs, **GENERATION_SETTINGS)
    torch.cuda.synchronize(DEVICE)
    elapsed = time.perf_counter() - started
    peak_allocated = torch.cuda.max_memory_allocated(DEVICE)
    peak_reserved = torch.cuda.max_memory_reserved(DEVICE)
    tokens = generated[0, 1:]  # Flan-T5 output begins with a decoder start token.
    length = int((tokens != processor.tokenizer.pad_token_id).sum())
    answer = processor.batch_decode(generated, skip_special_tokens=True)[0].strip()
    return {
        "question_id": int(question["question_id"]),
        "image_id": int(question["image_id"]),
        "answer": answer,
        "generation_seconds": elapsed,
        "generated_token_count": length,
        "allocated_before_bytes": allocated_before,
        "peak_allocated_bytes": peak_allocated,
        "peak_reserved_bytes": peak_reserved,
    }


def run_condition(condition):
    directory = EXPERIMENT_DIR / condition["name"]
    directory.mkdir(exist_ok=True)
    path = directory / "records.jsonl"
    records = read_records(path, question_ids, condition)
    remaining = questions[len(records):]
    try:
        if remaining:
            set_query_count(condition["k"], condition["seed"])
            torch.cuda.empty_cache()
            for question in questions[:WARMUP_QUESTIONS]:
                generate_answer(question)
            write_json_atomic(directory / "status.json", {"state": "running", "completed": len(records)})
            with path.open("a", encoding="utf-8") as stream:
                for question in tqdm(remaining, desc=condition["name"]):
                    record = generate_answer(question)
                    record.update(k=condition["k"], query_seed=condition["seed"])
                    stream.write(json.dumps(record, allow_nan=False) + "\n")
                    stream.flush()
                    os.fsync(stream.fileno())
                    records.append(record)
        write_json_atomic(directory / "predictions.json", [
            {"question_id": record["question_id"], "answer": record["answer"]} for record in records
        ])
        write_json_atomic(directory / "status.json", {"state": "complete", "completed": len(records)})
        print(condition["name"], "complete:", len(records), "questions")
    except BaseException as error:
        write_json_atomic(directory / "status.json", {
            "state": "interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
            "completed": len(records), "error": type(error).__name__ + ": " + str(error),
        })
        raise


# A single writer owns this experiment directory, including recovery and exports.
with FileLock(str(EXPERIMENT_DIR / "experiment.lock"), timeout=0):
    manifest_path = EXPERIMENT_DIR / "manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest["signature"] != signature or manifest["experiment"] != experiment:
            raise ValueError("The saved manifest differs from this experiment.")
    else:
        write_json_atomic(manifest_path, {
            "signature": signature, "created_utc": datetime.now(timezone.utc).isoformat(),
            "experiment": experiment,
        })
    try:
        for condition in experiment["conditions"]:
            run_condition(condition)
        write_json_atomic(EXPERIMENT_DIR / "complete.json", {
            "signature": signature, "condition_count": len(conditions),
            "questions_per_condition": len(questions),
            "finished_utc": datetime.now(timezone.utc).isoformat(),
        })
    finally:
        restore_original_queries()
        torch.cuda.empty_cache()

print("Finished. Original K=32 queries restored. Run score_results.ipynb next.")
